# 02 · Model training
Stratified train/test split (**artifact 4**), 5-fold cross-validated comparison of a logistic-regression baseline and gradient boosting **on the training set only**, then fits both and saves them (**artifact 5**). The test set stays untouched until notebook 03.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

import pipeline_config as cfg

df = pd.read_csv(cfg.CLEAN_FILE)
features = cfg.feature_columns(df)
print(f"{len(df):,} customers | {len(features)} features")
print("Excluded:", cfg.EXCLUDED)

## 1. Stratified train / test split

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[cfg.TARGET], random_state=cfg.RANDOM_STATE)
print(f"train {len(train_df):,} ({train_df[cfg.TARGET].mean():.1%} positive) | "
      f"test {len(test_df):,} ({test_df[cfg.TARGET].mean():.1%} positive)")

split = pd.concat([train_df[["customer_id"]].assign(split="train"),
                   test_df[["customer_id"]].assign(split="test")])
split.to_csv(cfg.SPLIT_FILE, index=False)

## 2. Cross-validated comparison (training set)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.RANDOM_STATE)
X_train, y_train = train_df[features], train_df[cfg.TARGET]
candidates = {"baseline_logreg": cfg.build_baseline(df),
              "gradient_boosting": cfg.build_gradient_boosting(df)}

cv_rows = []
for name, model in candidates.items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision"],
                       error_score="raise")
    cv_rows.append({"model": name, "cv_roc_auc": s["test_roc_auc"].mean(),
                    "cv_pr_auc": s["test_average_precision"].mean()})
cv_table = pd.DataFrame(cv_rows).set_index("model").round(3)
cv_table

In [ ]:
ax = cv_table.rename(columns={"cv_roc_auc": "ROC-AUC", "cv_pr_auc": "PR-AUC"}).plot.bar(
    figsize=(6, 3.8), rot=0, color=["#2f6db5", "#c25b12"], ylim=(0, 1.15))
ax.axhline(train_df[cfg.TARGET].mean(), color="grey", linestyle="--", linewidth=1, label="random PR-AUC")
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.2f}", ha="center", va="bottom")
ax.set(title="5-fold cross-validation on the training set", xlabel="")
ax.legend(loc="upper center", ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(cfg.CHART_DIR / "04_cv_comparison.png", dpi=150)
plt.show()

## 3. Fit on the full training set and save

In [ ]:
cfg.MODEL_DIR.mkdir(exist_ok=True)
for name, model in candidates.items():
    model.fit(X_train, y_train)
    joblib.dump(model, cfg.MODEL_DIR / f"{name}.joblib")
    print(f"saved {name}.joblib")
print("boosting iterations after early stopping:", candidates["gradient_boosting"][-1].n_iter_)